# Long-Term Time Series Forecasting (LTSF) with Exogenous Weather Features
### Models: Google's TSMixer (All-MLP) vs. Weather-Aware iTransformer

This notebook demonstrates how integrating local weather covariates (Temperature, Humidity, WindSpeed) can significantly boost electricity forecasting accuracy ($R^2$) for medium-to-long term horizons (**1-week / 168h** and **1-month / 720h**).

We compare two SOTA architectures:
1. **Google Research's TSMixer (2023)**: An All-MLP architecture that mixes temporal and feature representations using simple linear layers, showing exceptional efficiency ($O(L)$ time-complexity) and cross-variable tracking.
2. **Weather-Aware iTransformer**: An inversion of the traditional Transformer that treats weather covariates and target metrics as distinct variable tokens, applying self-attention across variables rather than time steps.

### Data Route Integration:
- We merge the 1-minute **UCI Electric Power Consumption (IHEPC)** dataset with the hourly **Clamart Weather** dataset (2006-2010).
- Targets are scaled locally using `RevIN`. Exogenous features are scaled using a training-fitted `StandardScaler` to prevent leakage.

In [ ]:
import os
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Devices Count: {torch.cuda.device_count()}")

## 1. Reversible Instance Normalization (RevIN)

In [ ]:
class RevIN(nn.Module):
    """Reversible Instance Normalization (stateless across forward passes for multi-GPU safety)"""
    def __init__(self, num_features: int, eps=1e-5, affine=True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(self.num_features))
            self.affine_bias = nn.Parameter(torch.zeros(self.num_features))

    def forward(self, x, mode: str, mean=None, stdev=None):
        if mode == 'norm':
            mean = torch.mean(x, dim=1, keepdim=True).detach()
            stdev = torch.sqrt(torch.var(x, dim=1, keepdim=True, unbiased=False) + self.eps).detach()
            x = (x - mean) / stdev
            if self.affine:
                x = x * self.affine_weight + self.affine_bias
            return x, mean, stdev
        elif mode == 'denorm':
            if self.affine:
                x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            x = x * stdev + mean
            return x

## 2. Google's TSMixer Implementation
TSMixer mixes step information along the lookback window (time mixing) and correlations across variables (feature mixing).

In [ ]:
class TSMixerBlock(nn.Module):
    """Mixer block: Time mixing and Feature mixing"""
    def __init__(self, c_in, lookback, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm([lookback, c_in])
        self.time_mlp = nn.Sequential(
            nn.Linear(lookback, lookback),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm([lookback, c_in])
        self.feature_mlp = nn.Sequential(
            nn.Linear(c_in, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, c_in),
            nn.Dropout(dropout)
        )
        
    def forward(self, x):
        # Time Mixing (mix across sequence length L)
        res = x
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = self.time_mlp(x)
        x = x.transpose(1, 2)
        x = x + res
        
        # Feature Mixing (mix across channels C)
        res = x
        x = self.norm2(x)
        x = self.feature_mlp(x)
        x = x + res
        return x

class TSMixer(nn.Module):
    """Google's TSMixer with Exogenous Weather Feature Support"""
    def __init__(self, c_out=7, c_weather=3, lookback=336, forecast_horizon=168, 
                 d_ff=64, num_blocks=3, dropout=0.1, revin=True):
        super().__init__()
        self.c_out = c_out
        self.c_weather = c_weather
        self.c_in = c_out + c_weather
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        
        self.revin = revin
        if self.revin:
            self.revin_layer = RevIN(c_out)
            
        self.blocks = nn.ModuleList([
            TSMixerBlock(self.c_in, lookback, d_ff, dropout)
            for _ in range(num_blocks)
        ])
        self.head = nn.Linear(lookback * self.c_in, forecast_horizon * c_out)
        
    def forward(self, x_targets, x_weather, temporal=None):
        # x_targets: (B, L, C_out)
        # x_weather: (B, L, C_weather)
        
        if self.revin:
            x_targets, mean, stdev = self.revin_layer(x_targets, 'norm')
            
        # Concatenate target channels and weather channels
        x = torch.cat([x_targets, x_weather], dim=-1) # (B, L, C_in)
        
        for block in self.blocks:
            x = block(x)
            
        # Predict and map to target columns only
        batch_size = x.size(0)
        x_flat = x.reshape(batch_size, -1)
        out = self.head(x_flat)
        out = out.reshape(batch_size, self.forecast_horizon, self.c_out)
        
        if self.revin:
            out = self.revin_layer(out, 'denorm', mean=mean, stdev=stdev)
        return out

## 3. Weather-Aware iTransformer Implementation
Treats targets and weather covariates as distinct tokens and maps relationships across channels.

In [ ]:
class WeatheriTransformer(nn.Module):
    """Weather-Aware iTransformer with weather variables as transposed tokens"""
    def __init__(self, c_out=7, c_weather=3, lookback=336, forecast_horizon=168, 
                 d_model=128, n_heads=8, n_layers=3, d_ff=256, 
                 dropout=0.1, revin=True, affine=True):
        super().__init__()
        self.c_out = c_out
        self.c_weather = c_weather
        self.c_in = c_out + c_weather
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        
        self.revin = revin
        if self.revin:
            self.revin_layer = RevIN(c_out, affine=affine)
            
        self.token_embedding = nn.Linear(lookback, d_model)
        self.weather_embedding = nn.Linear(lookback, d_model)
        
        self.hour_embed = nn.Embedding(24, d_model)
        self.weekday_embed = nn.Embedding(7, d_model)
        self.month_embed = nn.Embedding(12, d_model)
        self.temporal_pool = nn.Linear(lookback, 1)
        
        self.channel_embed = nn.Parameter(torch.zeros(1, self.c_in, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, forecast_horizon)
        
    def forward(self, x_targets, x_weather, temporal):
        # x_targets: (B, L, C_out)
        # x_weather: (B, L, C_weather)
        # temporal: (B, L, 3)
        
        if self.revin:
            x_targets, mean, stdev = self.revin_layer(x_targets, 'norm')
            
        targets_trans = x_targets.permute(0, 2, 1) # (B, C_out, L)
        weather_trans = x_weather.permute(0, 2, 1) # (B, C_weather, L)
        
        enc_targets = self.token_embedding(targets_trans)
        enc_weather = self.weather_embedding(weather_trans)
        enc_in = torch.cat([enc_targets, enc_weather], dim=1) # (B, C_in, d_model)
        
        hour_emb = self.hour_embed(temporal[:, :, 0])
        weekday_emb = self.weekday_embed(temporal[:, :, 1])
        month_emb = self.month_embed(temporal[:, :, 2])
        temp_emb = hour_emb + weekday_emb + month_emb
        temp_emb_pooled = self.temporal_pool(temp_emb.permute(0, 2, 1)).squeeze(-1)
        
        enc_in = enc_in + temp_emb_pooled.unsqueeze(1)
        enc_in = enc_in + self.channel_embed
        
        # Self-Attention across channels
        enc_out = self.encoder(enc_in)
        
        # Retain targets only for forecast projection
        enc_out_targets = enc_out[:, :self.c_out, :]
        dec_out = self.head(enc_out_targets)
        dec_out = dec_out.permute(0, 2, 1) # (B, H, C_out)
        
        if self.revin:
            dec_out = self.revin_layer(dec_out, 'denorm', mean=mean, stdev=stdev)
        return dec_out

## 4. Exogenous Data Preprocessing & PyTorch Dataloaders

In [ ]:
class ExogenousEnergyDataset(Dataset):
    def __init__(self, targets, weather, stamps, lookback, horizon):
        self.targets = targets
        self.weather = weather
        self.stamps = stamps
        self.lookback = lookback
        self.horizon = horizon

    def __len__(self):
        return len(self.targets) - self.lookback - self.horizon + 1

    def __getitem__(self, idx):
        x_t = self.targets[idx : idx + self.lookback]
        x_w = self.weather[idx : idx + self.lookback]
        y_t = self.targets[idx + self.lookback : idx + self.lookback + self.horizon]
        
        stamp_x = self.stamps[idx : idx + self.lookback]
        hour = stamp_x.hour.values
        dayofweek = stamp_x.dayofweek.values
        month = stamp_x.month.values - 1
        temporal = np.stack([hour, dayofweek, month], axis=1)
        
        return (torch.tensor(x_t, dtype=torch.float32),
                torch.tensor(x_w, dtype=torch.float32),
                torch.tensor(y_t, dtype=torch.float32),
                torch.tensor(temporal, dtype=torch.long))

def prepare_weather_dataloaders(targets_csv_path, weather_csv_path, lookback, horizon, batch_size=32):
    print(f"Loading energy targets from {targets_csv_path}...")
    df_t = pd.read_csv(targets_csv_path, sep=';', na_values=['?'], dtype={'Date': str, 'Time': str})
    
    print("Parsing datetime indexes...")
    df_t['Datetime'] = pd.to_datetime(df_t['Date'] + ' ' + df_t['Time'], format='%d/%m/%Y %H:%M:%S')
    df_t = df_t.drop(columns=['Date', 'Time']).set_index('Datetime')
    
    target_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    df_t[target_cols] = df_t[target_cols].astype(float).ffill().bfill()
    
    print("Resampling energy data to hourly frequency...")
    df_t_hourly = df_t[target_cols].resample('h').mean().ffill().bfill()
    
    print(f"Loading weather variables from {weather_csv_path}...")
    df_w = pd.read_csv(weather_csv_path, parse_dates=['Datetime']).set_index('Datetime')
    weather_cols = ['Temperature', 'Humidity', 'WindSpeed']
    df_w = df_w[weather_cols].ffill().bfill()
    
    print("Merging energy and weather covariates...")
    df_merged = df_t_hourly.join(df_w, how='inner').ffill().bfill()
    print(f"Merged hourly dataset has {len(df_merged)} total records.")
    
    split_idx = int(len(df_merged) * 0.8)
    train_df = df_merged.iloc[:split_idx]
    val_df = df_merged.iloc[split_idx:]
    
    # Scale targets globally (fit strictly on train)
    print("Fitting target scaler...")
    target_scaler = StandardScaler()
    train_t_scaled = target_scaler.fit_transform(train_df[target_cols].values)
    val_t_scaled = target_scaler.transform(val_df[target_cols].values)
    
    # Scale weather globally (fit strictly on train)
    print("Fitting weather scaler...")
    weather_scaler = StandardScaler()
    train_w_scaled = weather_scaler.fit_transform(train_df[weather_cols].values)
    val_w_scaled = weather_scaler.transform(val_df[weather_cols].values)
    
    train_stamps = train_df.index
    val_stamps = val_df.index
    
    train_dataset = ExogenousEnergyDataset(train_t_scaled, train_w_scaled, train_stamps, lookback, horizon)
    val_dataset = ExogenousEnergyDataset(val_t_scaled, val_w_scaled, val_stamps, lookback, horizon)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, target_scaler, weather_scaler

## 5. Training & Evaluation Functions

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda', save_path='model.pth'):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x_t, x_w, y_t, temp in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            x_t, x_w, y_t, temp = x_t.to(device), x_w.to(device), y_t.to(device), temp.to(device)
            optimizer.zero_grad()
            preds = model(x_t, x_w, temp)
            loss = criterion(preds, y_t)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_t, x_w, y_t, temp in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                x_t, x_w, y_t, temp = x_t.to(device), x_w.to(device), y_t.to(device), temp.to(device)
                preds = model(x_t, x_w, temp)
                val_loss += criterion(preds, y_t).item()
                
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), save_path)
            else:
                torch.save(model.state_dict(), save_path)
            print(f"--> Saved best model weights to {save_path}")

def evaluate_model(model, val_loader, scaler, device, target_idx=0, model_label="Model"):
    model.eval()
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for x_t, x_w, y_t, temp in tqdm(val_loader, desc=f"[Evaluating {model_label}]"):
            x_t, x_w, y_t, temp = x_t.to(device), x_w.to(device), y_t.to(device), temp.to(device)
            preds = model(x_t, x_w, temp)
            
            all_preds.append(preds.cpu().numpy())
            all_trues.append(y_t.cpu().numpy())
            
    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)
    
    N, H, C = all_preds.shape
    preds_flat = all_preds.reshape(-1, C)
    trues_flat = all_trues.reshape(-1, C)
    
    # Inverse scaling
    preds_orig = scaler.inverse_transform(preds_flat).reshape(N, H, C)
    trues_orig = scaler.inverse_transform(trues_flat).reshape(N, H, C)
    
    preds_gap = preds_orig[:, :, target_idx].flatten()
    trues_gap = trues_orig[:, :, target_idx].flatten()
    
    mae = mean_absolute_error(trues_gap, preds_gap)
    r2 = r2_score(trues_gap, preds_gap)
    
    print(f"\n--- Evaluation for Global_active_power ({model_label}) ---")
    print(f"Actual MAE (Original Scale): {mae:.4f} kW")
    print(f"R-squared: {r2:.4f}\n")
    return mae, r2, preds_orig, trues_orig

## 6. Execution: Training TSMixer and Weather-Aware iTransformer

In [ ]:
def find_file(filename, search_dir='/kaggle/input'):
    if not os.path.exists(search_dir):
        return None
    for root, dirs, files in os.walk(search_dir):
        if filename in files:
            path = os.path.join(root, filename)
            print(f"Found {filename} at: {path}")
            return path
    return None

TARGETS_PATH = find_file('household_power_consumption.txt')
if TARGETS_PATH is None:
    TARGETS_PATH = 'c:/Users/salah/Documents/MASTER/PFE2/data/household_power_consumption.txt'
    if not os.path.exists(TARGETS_PATH):
        TARGETS_PATH = '../data/household_power_consumption.txt'
        if not os.path.exists(TARGETS_PATH):
            TARGETS_PATH = 'data/household_power_consumption.txt'

WEATHER_PATH = find_file('clamart_weather_2006_2010.csv')
if WEATHER_PATH is None:
    WEATHER_PATH = 'c:/Users/salah/Documents/MASTER/PFE2/data/clamart_weather_2006_2010.csv'
    if not os.path.exists(WEATHER_PATH):
        WEATHER_PATH = '../data/clamart_weather_2006_2010.csv'
        if not os.path.exists(WEATHER_PATH):
            WEATHER_PATH = 'data/clamart_weather_2006_2010.csv'

print(f"Resolved TARGETS_PATH: {TARGETS_PATH}")
print(f"Resolved WEATHER_PATH: {WEATHER_PATH}")

HORIZONS = [
    {"name": "1_week", "horizon": 168, "lookback": 512},
    {"name": "1_month", "horizon": 720, "lookback": 1440}
]

results = {}
predictions_store = {}

for config in HORIZONS:
    horizon_name = config['name']
    results[horizon_name] = {}
    predictions_store[horizon_name] = {}
    
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {horizon_name} (Horizon: {config['horizon']}h, Lookback: {config['lookback']}h)")
    print(f"{'='*60}")
    
    train_loader, val_loader, target_scaler, weather_scaler = prepare_weather_dataloaders(
        TARGETS_PATH, WEATHER_PATH, lookback=config['lookback'], horizon=config['horizon'], batch_size=64
    )
    
    for model_name in ["TSMixer", "WeatheriTransformer"]:
        print(f"\n{'-'*40}")
        print(f"Training {model_name} for horizon {horizon_name}...")
        print(f"{'-'*40}")
        
        if model_name == "TSMixer":
            model = TSMixer(
                c_out=7, c_weather=3, lookback=config['lookback'], forecast_horizon=config['horizon'],
                d_ff=128, num_blocks=3, dropout=0.2, revin=True
            )
        else:
            model = WeatheriTransformer(
                c_out=7, c_weather=3, lookback=config['lookback'], forecast_horizon=config['horizon'],
                d_model=128, n_heads=8, n_layers=3, d_ff=256, dropout=0.2, revin=True
            )
            
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
            model = nn.DataParallel(model)
            
        save_path = f"{model_name.lower()}_weather_{horizon_name}_weights.pth"
        
        # Train model
        train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, device=device, save_path=save_path)
        
        # Safe load
        print(f"Loading best model weights from {save_path}...")
        state_dict = torch.load(save_path, map_location=device)
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(state_dict)
        else:
            model.load_state_dict(state_dict)
            
        # Evaluate model
        mae, r2, preds_orig, trues_orig = evaluate_model(
            model, val_loader, target_scaler, device, target_idx=0, model_label=f"{model_name}_{horizon_name}"
        )
        results[horizon_name][model_name] = {"MAE": mae, "R2": r2}
        predictions_store[horizon_name][model_name] = preds_orig
        
    predictions_store[horizon_name]["GroundTruth"] = trues_orig

## 7. Results Comparison & Forecasting Plots

In [ ]:
print("\n" + "="*60)
print("WEATHER INTEGRATION EXPERIMENT RESULTS")
print("="*60)
for horizon, models in results.items():
    print(f"\nHorizon: {horizon}")
    for model_name, metrics in models.items():
        print(f"  {model_name}: MAE = {metrics['MAE']:.4f} kW, R2 = {metrics['R2']:.4f}")

for config in HORIZONS:
    horizon_name = config['name']
    horizon_len = config['horizon']
    
    # Select validation sample 42 for forecast visualization
    sample_idx = 42
    
    true_seq = predictions_store[horizon_name]["GroundTruth"][sample_idx, :, 0]
    tsmixer_seq = predictions_store[horizon_name]["TSMixer"][sample_idx, :, 0]
    itrans_seq = predictions_store[horizon_name]["WeatheriTransformer"][sample_idx, :, 0]
    
    plt.figure(figsize=(15, 6))
    plt.plot(true_seq, label="Actual (Ground Truth)", color="black", linewidth=2.5)
    plt.plot(tsmixer_seq, label="TSMixer Forecast (with Weather)", color="#2a9d8f", linestyle="--", linewidth=2)
    plt.plot(itrans_seq, label="Weather-iTransformer Forecast", color="#f4a261", linestyle="-.", linewidth=2)
    
    plt.title(f"Forecast Comparison with Weather Features: {horizon_name} ({horizon_len}h)", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Hour Index", fontsize=12)
    plt.ylabel("Global Active Power (kW)", fontsize=12)
    plt.legend(fontsize=11, loc="upper right")
    plt.tight_layout()
    plt.show()